In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone repo
import os, shutil, sys

os.chdir('/content')  # ensure we're not inside the dir we're about to delete

if os.path.exists('/content/myViT'):
    shutil.rmtree('/content/myViT')

from google.colab import userdata
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_URL = 'https://github.com/dicarlo-a/myViT.git'
auth_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')
!git clone {auth_url} /content/myViT

sys.path.insert(0, '/content/myViT')
os.chdir('/content/myViT')

In [ ]:
# 3. Install extra dependencies + symlink runs folder from Drive
!pip install -q sentence-transformers datasets einops accelerate

import os

runs_link = '/content/myViT/runs'
drive_runs = '/content/drive/MyDrive/Caltech/Junior/Spring/EE CNS CS 148b/HW3/runs'

os.makedirs(drive_runs, exist_ok=True)
if os.path.islink(runs_link) or os.path.exists(runs_link):
    shutil.rmtree(runs_link) if os.path.isdir(runs_link) else os.remove(runs_link)
os.symlink(drive_runs, runs_link)
print(f"Runs linked: {os.listdir(drive_runs)}")

In [ ]:
# 4. Run CLIP pretraining
%run /content/myViT/scripts/pretrain_clip.py --config /content/myViT/configs/clip_eurosat.yaml --device cuda

In [ ]:
# 5. LoRA comparison: linear probe vs. LoRA (r=8) vs. full fine-tuning
%run /content/myViT/scripts/finetune_resisc.py --config /content/myViT/configs/lora_resisc.yaml --compare-all --pretrained /content/myViT/runs/clip_eurosat/best.pt --device cuda

In [ ]:
# 6. LoRA rank sweep: r in {1,2,4,8,16,32,64} with alpha=2r
%run /content/myViT/scripts/finetune_resisc.py --config /content/myViT/configs/lora_resisc.yaml --rank-sweep --pretrained /content/myViT/runs/clip_eurosat/best.pt --device cuda